# Imitation Learning

In [ ]:
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio
import plotly.graph_objs as go
from plotly.subplots import make_subplots


def line(error_y_mode=None, **kwargs):
    """Extension of `plotly.express.line` to use error bands."""
    error_modes = {"bar", "band", "bars", "bands", None}

    if error_y_mode not in error_modes:
        raise ValueError(
            f"'error_y_mode' must be one of {error_modes}, received"
            + f" {repr(error_y_mode)}."
        )

    if error_y_mode in {"bar", "bars", None}:
        fig = px.line(**kwargs)
    elif error_y_mode in {"band", "bands"}:
        if "error_y" not in kwargs:
            raise ValueError(
                "If you provide 'error_y_mode' you must also provide 'error_y'."
            )

        kwargs_per_line = [kwargs]

        if (
            kwargs["data_frame"] is not None
            and kwargs["y"] is not None
            and not isinstance(kwargs["y"], str)
        ):
            kwargs_per_line = []

            for i in range(len(kwargs["y"])):
                line_kwargs = kwargs.copy()
                line_kwargs["y"] = line_kwargs["y"][i]
                line_kwargs["error_y"] = line_kwargs["error_y"][i]
                line_kwargs["error_y_minus"] = line_kwargs["error_y_minus"][i]
                kwargs_per_line.append(line_kwargs)

        fig = px.line(
            **{
                arg: val
                for arg, val in kwargs.items()
                if arg not in ("error_y", "error_y_minus")
            }
        )

        for i, line_kwargs in enumerate(kwargs_per_line):
            data = px.line(**line_kwargs).data[0]
            x = list(data["x"])
            y_upper = list(data["y"] + data["error_y"]["array"])
            y_lower = list(
                data["y"] - data["error_y"]["array"]
                if data["error_y"]["arrayminus"] is None
                else data["y"] - data["error_y"]["arrayminus"]
            )

            colors = fig.data[i]["line"]["color"]
            colors = tuple(int(colors.lstrip("#")[j : j + 2], 16) for j in (0, 2, 4))
            color_str = f"rgba({colors},.3)"
            color_str = color_str.replace("((", "(").replace("),", ",").replace(" ", "")

            fig.add_trace(
                go.Scatter(
                    x=x + x[::-1],
                    y=y_upper + y_lower[::-1],
                    fill="toself",
                    fillcolor=color_str,
                    line=dict(color="rgba(255,255,255,0)"),
                    hoverinfo="skip",
                    showlegend=False,
                    legendgroup=data["legendgroup"],
                    xaxis=data["xaxis"],
                    yaxis=data["yaxis"],
                )
            )

        # Reorder data as said here: https://stackoverflow.com/a/66854398/8849755
        reordered_data = []

        for i in range(int(len(fig.data) / 2)):
            reordered_data.append(fig.data[i + int(len(fig.data) / 2)])
            reordered_data.append(fig.data[i])

        fig.data = tuple(reordered_data)

    return fig


pio.templates.default = "plotly_white"
default_layout = dict(
    font_family="FiraMono Nerd Font",
    font_color="#15244c",
    font_size=34,
    legend=dict(
        yanchor="bottom",
        y=0.01,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(0,0,0,0)"
    ),
    margin=dict(l=1, t=1, b=1, r=1)
)

default_xaxes = dict(
    showline=True,
    linecolor="#aeb6c2",
    linewidth=2,
    row=1,
    col=1,
    mirror=True,
    gridcolor="#e6ebeb",
)

default_yaxes = dict(
    showline=True,
    linecolor="#aeb6c2",
    linewidth=2,
    row=1,
    col=1,
    mirror=True,
    gridcolor="#e6ebeb",
    ticksuffix=" ",
    rangemode="tozero",
)

## Load Runs

In [ ]:
from collections.abc import MutableMapping
import pathlib as pl
import time

import json
import numpy as np
import wandb


def load_runs(folder_path):
    run_files = pl.Path(folder_path).glob("*.json")
    task = folder_path.rstrip("/").rsplit("/", 1)[-1]
    runs = {}

    for file in run_files:
        with open(file, "r") as file:
            data = json.load(file)
            
        seed = int(data["config"]["arch"]["seed"])
        expansion_steps =  int(data["config"]["system"]["expansion_steps"])
        beta = float(data["config"]["system"]["beta"])
        gamma = float(data["config"]["system"]["gamma"])
        h = int(data["config"]["system"]["h"])
        s_prob = float(data["config"]["env"]["kwargs"]["stuck_prob"])
        us_prob = float(data["config"]["env"]["kwargs"]["unstuck_prob"])
        len_dt = len(eval(data["config"]["env"]["kwargs"]["difficult_terrain"]))
        expl_type = data["config"]["system"]["bonus"]["bonus_type"]
        expl_coeff = None
        expl_coeff = float(data["config"]["system"]["bonus"]["kw"]["coeff"])
        expl_kw = f"-coef={expl_coeff}"
        tag = f"occ/gamma={gamma}/h={h}/beta={beta}/expl={expl_type}{expl_kw}/expansion={expansion_steps}"
        tag += f"/stuck_prob={s_prob}/unstuck_prob={us_prob}/dt={len_dt}"

        if tag not in runs:
            runs[tag] = {"seed": [], "costs": [], "erm": [], "states": [], "actions": []}
            runs[tag]["beta"] = beta
            runs[tag]["expl"] = expl_type
            runs[tag]["expl_coeff"] = expl_coeff
            runs[tag]["expansion"] = expansion_steps

        cost = data["TRAIN"]["cost"][0]
        erm = data["TRAIN"]["erm"]
        states = data["TRAIN"]["state"]
        actions = data["TRAIN"]["action"]
        assert isinstance(cost, float)
        assert len(erm) == len(actions) == h
        assert len(states) == len(actions) + 1
        assert all([(e is not None) for e in erm])
        assert all([(s is not None) for s in states])
        assert all([(a is not None) for a in actions])
        assert isinstance(cost, float)
        runs[tag]["seed"].append(seed)
        runs[tag]["costs"].append(cost)
        runs[tag]["erm"].append(erm)
        runs[tag]["states"].append(states)
        runs[tag]["actions"].append(actions)
        
    for tag in runs:
        runs[tag]["seed"] = np.array(runs[tag]["seed"])
        runs[tag]["costs"] = np.array(runs[tag]["costs"])
        runs[tag]["erm"] = np.stack(runs[tag]["erm"], axis=0)
        runs[tag]["states"] = np.stack(runs[tag]["states"], axis=0)
        runs[tag]["actions"] = np.stack(runs[tag]["actions"], axis=0)

    return runs
    
def _flatten(dictionary, parent_key='', separator='/'):
    items = []

    for key, value in dictionary.items():
        new_key = parent_key + separator + key if parent_key else key
        
        if isinstance(value, MutableMapping):
            items.extend(_flatten(value, new_key, separator=separator).items())
        else:
            items.append((new_key, value))

    return dict(items)

data_keys = ["seed", "costs", "erm", "states", "actions"]
runs = load_runs("../results/erm_occupancy_mcts/99/0.99/il")  # NOTE: Change this to match with the hyper-params used

# Check that there are no repeated seed
all_seeds = [runs[r]["seed"] for r in runs]
all_seeds = np.concatenate(all_seeds).tolist()
assert len(all_seeds) == len(set(all_seeds)), "Repeated seeds"

data = _flatten({t: {k: runs[t][k] for k in data_keys} for t in runs})

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


df = {"erm": np.array([]), "gamma": [], "h": [], "beta": [], "policy": [], "bonus": [], "stuck_prob": [], "unstuck_prob": [], "dt": []}

for k, arr in data.items():
    if "/costs" not in k:
        continue

    gamma = float(k.split("/")[1].split("=", 1)[1])
    h = int(k.split("/")[2].split("=", 1)[1])
    beta = float(k.split("/")[3].split("=", 1)[1])
    expansion = int(k.split("/")[5].split("=", 1)[1])
    bonus = k.split("/")[4].split("-", 1)[0].split("=", 1)[1]
    stuck_prob = float(k.split("/")[6].split("=", 1)[1])
    unstuck_prob = float(k.split("/")[7].split("=", 1)[1])
    dt = int(k.split("/")[8].split("=", 1)[1])
    df["erm"] = np.concatenate((df["erm"], arr), axis=0)
    df["gamma"] += [gamma] * arr.shape[0]
    df["h"] += [h] * arr.shape[0]
    df["beta"] += [beta] * arr.shape[0]
    df["policy"] += [f"MCTS ({expansion})"] * arr.shape[0]
    df["bonus"] += [bonus] * arr.shape[0]
    df["stuck_prob"] += [stuck_prob] * arr.shape[0]
    df["unstuck_prob"] += [unstuck_prob] * arr.shape[0]
    df["dt"] += [dt] * arr.shape[0]

df = pd.DataFrame(df)
df["beta"] = df["beta"].round(3)
betas = [str(beta) for beta in np.sort(df["beta"].unique())]
policies = set(df["policy"].unique())

## Plots

In [ ]:
_df = df.copy()

# NOTE: Change this to match with the hyper-params used
_df = _df[_df["gamma"] == 0.99]
_df = _df[_df["bonus"] == "erm_uct"]
_df = _df[(_df["stuck_prob"] == 0.1) & (_df["unstuck_prob"] == 0.01) & (_df["dt"] == 20)]
_df["beta"] = _df["beta"].astype(str)
fig = px.box(_df, x="beta", y="erm", category_orders={"policy": policies, "beta": betas})

_default_layout = default_layout | dict(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=0.99,
    bgcolor="rgba(0,0,0,0)"
))

fig.update_traces(boxmean=True)
fig.update_layout(legend_title_text="algorithm", boxmode="group", **_default_layout)
fig.update_xaxes(title="Beta", **default_xaxes)
fig.update_yaxes(title="Cost", **default_yaxes)
time.sleep(2)
fig.write_image("imgs/il_cost_dist.pdf")
time.sleep(2)
fig.show()

In [ ]:
import yaml


# get difficult terrain positions - `dt_pos`
with open('../src/risk_aware_gumdp/configs/env/il.yaml', 'r') as file:
    cfg = yaml.safe_load(file)

grid_size = cfg["kwargs"]["size"]
dt_pos = cfg["kwargs"]["difficult_terrain"]
dt_pos = [x * grid_size[1] + y for (x, y) in dt_pos]
dt_pos = np.array(dt_pos)

In [ ]:
behavioral_seq = [91, 92, 93, 83, 82, 81, 80, 70, 71, 72, 73, 63, 62, 61, 60, 50, 40, 30, 20, 10, 0, 1, 11, 21, 31, 41, 51, 52, 42, 32, 22, 12, 2, 3, 13, 23, 33, 43, 53, 54, 44, 34, 24, 14, 4, 5, 15, 25, 35, 45, 55, 56, 46, 36, 26, 16, 6, 7, 17, 27, 37, 47, 57, 58, 48, 38, 28, 18, 8, 9, 19, 29, 39, 49, 59, 69, 68, 67, 66, 65, 64, 74, 75, 76, 77, 78, 79, 89, 88, 87, 86, 85, 84, 94, 95, 96, 97, 98, 99]
behavioral_seq = np.array(behavioral_seq)
behavioral_seq = np.stack((behavioral_seq // grid_size[1], behavioral_seq % grid_size[1]), axis=1)

In [ ]:
# AVG TRAPPED RUNS BY TIME

import numpy as np
import yaml


# define hyper-params to filter runs
# NOTE: Change this to match with the hyper-params used
hyper_params = {"gamma": 0.99, "bonus": "erm_uct", "stuck_prob": 0.1, "unstuck_prob": 0.01, "dt": 20}

# for each run count number of times the agent visited difficult terrain
count = {}

for tag in data:
    gamma = float(tag.split("/")[1].split("=", 1)[1])
    bonus = tag.split("/")[4].split("-", 1)[0].split("=", 1)[1]
    stuck_prob = float(tag.split("/")[6].split("=", 1)[1])
    unstuck_prob = float(tag.split("/")[7].split("=", 1)[1])
    dt = int(tag.split("/")[8].split("=", 1)[1])
    data_type = tag.split("/")[-1]
    cond = gamma == hyper_params["gamma"]
    cond = cond and bonus == hyper_params["bonus"]
    cond = cond and stuck_prob == hyper_params["stuck_prob"]
    cond = cond and unstuck_prob == hyper_params["unstuck_prob"]
    cond = cond and dt == hyper_params["dt"]
    cond = cond and data_type == "states"
    
    if not cond:
        continue

    states = data[tag][:, 1:]  # this is a `np.ndarray`
    mask = np.isin(states, dt_pos)
    count[tag] = np.sum(mask, axis=0)

# create custom dataframe with visit information
count_df = {"count": np.array([]), "time": [], "beta": [], "policy": []}

for tag in count:
    beta = float(tag.split("/")[3].split("=", 1)[1])
    expansion = int(tag.split("/")[5].split("=", 1)[1])
    count_df["count"] = np.concatenate((count_df["count"], count[tag]), axis=0)
    count_df["time"] += list(range(count[tag].shape[0]))
    count_df["beta"] += [beta] * count[tag].shape[0]
    count_df["policy"] += [f"{beta}"] * count[tag].shape[0]

count_df = pd.DataFrame(count_df)
count_df["beta"] = count_df["beta"].round(3)
count_df["beta"] = count_df["beta"].astype(str)
policies = [f"0.002", f"1.0", f"500.0"]

# plot visit count data
fig = px.histogram(
    count_df,
    x="time",
    y="count",
    color="policy",
    histfunc="avg",
    nbins=20,
    barmode="group",
    category_orders={"policy": policies, "beta": betas}
)

_default_layout = default_layout | dict(legend=dict(
    orientation="h",
    yanchor="bottom",
    y=1.02,
    xanchor="right",
    x=1,
    font_size=24
), font_size=24)
_default_layout["margin"]["b"] = 100

fig.update_layout(legend_title_text="Beta", bargap=0.2, **_default_layout)
fig.update_xaxes(title="Timestep", **default_xaxes)
fig.update_yaxes(title="Avg. nr. of runs", **default_yaxes)
time.sleep(2)
fig.write_image("imgs/il_visited_dt.pdf")
time.sleep(2)
fig.show()